# Short Story Studio — Colab Notebook

A notebook-native creator workspace for Short Story Speaker. Use this inside Colab; it does **not** start a separate web server. Your older laptop only edits the notebook/script locally; AI model weights stay in the temporary Colab runtime.

## Runtime choices
- **CPU motion graphics**: no GPU required; produces animated text/graphic scenes using FFmpeg.
- **Chatterbox V3 Hindi voice**: optional model, loads only when you click Preview/Render. Use a clear 5–10 second recording of your own voice to clone it, or use the built-in voice.
- **AnimateDiff-Lightning AI footage**: optional, requires a Colab CUDA GPU with around 12 GB or more VRAM. It creates short portrait clips; the script assembles them into the final video.

Colab free GPU availability is not guaranteed. If no GPU is assigned, use CPU motion graphics and try again later. Colab notebooks are temporary: save your voice sample and exported videos to Google Drive. Keep the voice sample private and only use voices you own or have explicit permission to use. Review Colab's current policies before generating identity-matched voices.


In [ ]:
from pathlib import Path
import subprocess, sys
REPO = Path('/content/VIDEO-GENERATIONS-TOOLS')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/SaidulM/VIDEO-GENERATIONS-TOOLS.git', str(REPO)], check=True)
%cd /content/VIDEO-GENERATIONS-TOOLS
%pip -q install -r requirements.txt ipywidgets
print('Base studio ready. No AI model weights have been downloaded yet.')


In [ ]:
# Optional: mount Google Drive to keep voice samples and finished videos between Colab sessions.
from pathlib import Path
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as exc:
    print('Drive not mounted; outputs remain in this temporary Colab session.', exc)
if DRIVE_MOUNTED:
    PERSIST_ROOT = Path('/content/drive/MyDrive/ShortStoryStudio')
else:
    PERSIST_ROOT = Path('/content/short-story-studio-session')
VOICE_DIR = PERSIST_ROOT / 'voices'
EXPORT_DIR = PERSIST_ROOT / 'exports'
VOICE_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Voice profiles:', VOICE_DIR)
print('Exports:', EXPORT_DIR)


## Optional AI engines — install only when you need them
Run the next cell in a fresh Colab runtime. It installs the Chatterbox and Diffusers runtime packages, but **does not** download the large model checkpoints yet. The checkpoints download lazily when you first preview a voice or generate AI footage. The package pins PyTorch 2.6.0; Colab may ask you to restart the runtime after installation. If it does, reconnect and rerun the setup cells.


In [ ]:
%pip -q install -r requirements-colab.txt
print('Optional AI libraries installed. Model checkpoints will load on first use.')


In [ ]:
import json, os, shutil, uuid
from pathlib import Path
import ipywidgets as W
from IPython.display import display, Audio, Video, FileLink, Markdown, clear_output
import torch
from app.storyboard import create_storyboard
from app.renderer import render_video
from app.tts import synthesize
from app.providers import save_provider_settings
from app.config import OUTPUTS, UPLOADS

GPU_OK = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if GPU_OK else 'No CUDA GPU assigned'
GPU_VRAM = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if GPU_OK else 0
display(Markdown(f'**Runtime:** {GPU_NAME} · **VRAM:** {GPU_VRAM} GB · **AI video ready:** {GPU_OK and GPU_VRAM >= 12}'))

state = {'storyboard': None, 'voice_path': None, 'result': None, 'scene_widgets': []}
topic = W.Text(description='Topic', layout=W.Layout(width='100%'), placeholder='Example: A powerful Jumuah reminder')
script = W.Textarea(description='Script', layout=W.Layout(width='100%', height='170px'), placeholder='Paste your Hinglish/Hindi script here. Existing text is the source of truth.')
scene_notes = W.Textarea(description='Scene notes', layout=W.Layout(width='100%', height='85px'), placeholder='Optional: one visual direction per line')
style = W.Dropdown(description='Visual style', options=[('Cinematic documentary','cinematic'),('Islamic geometry','islamic_geometry'),('Cyberpunk','cyberpunk'),('Clay-like','clay'),('3D-inspired','3d'),('Minimal editorial','minimal')], value='cinematic')
project_format = W.Dropdown(description='Format', options=[('YouTube Shorts 9:16','short'),('Long video draft 16:9','long')], value='short')
language = W.Dropdown(description='Script language', options=[('Roman Hinglish','hinglish'),('Hindi','hindi'),('Bengali','bengali'),('English','english')], value='hinglish')
provider = W.Dropdown(description='LLM provider', options=[('Template only',''),('OpenRouter','openrouter'),('Groq','groq'),('DeepSeek','deepseek'),('Gemini','gemini'),('OpenAI','openai')], value='')
api_key = W.Password(description='API key', layout=W.Layout(width='100%'))
model_id = W.Text(description='Model ID', value='openrouter/free', layout=W.Layout(width='100%'))
draft_button = W.Button(description='Create storyboard', button_style='primary', icon='magic')
draft_output = W.Output()
scene_editor = W.VBox([])

voice_upload = W.FileUpload(accept='.wav,.mp3,.m4a,.flac,.ogg', multiple=False, description='Upload voice sample')
voice_mode = W.Dropdown(description='Voice', options=[('Chatterbox built-in voice','builtin'),('Clone my own voice (only where permitted)','reference'),('No narration','none')], value='builtin' if GPU_OK else 'none')
voice_consent = W.Checkbox(description='I own/have permission for this sample and will not make a deceptive impersonation', value=False)
exaggeration = W.FloatSlider(description='Expression', min=0.25, max=2.0, step=0.05, value=0.5, readout_format='.2f')
cfg_weight = W.FloatSlider(description='Guidance', min=0.0, max=1.0, step=0.05, value=0.5, readout_format='.2f')
temperature = W.FloatSlider(description='Variation', min=0.05, max=2.0, step=0.05, value=0.8, readout_format='.2f')
preview_button = W.Button(description='Preview Chatterbox voice', icon='play')
voice_output = W.Output()

visual_engine = W.Dropdown(description='Video engine', options=[('Lightweight motion graphics (CPU)','motion_cards'),('AnimateDiff-Lightning AI footage (GPU)','animatediff')], value='motion_cards', layout=W.Layout(width='100%'))
visual_engine.disabled = not (GPU_OK and GPU_VRAM >= 12)
duration = W.IntSlider(description='Target seconds', min=15, max=180, step=5, value=45)
burn_captions = W.Checkbox(description='Burn captions into video', value=True)
render_button = W.Button(description='Render YouTube video', button_style='success', icon='film', disabled=True)
render_output = W.Output()

def _uploaded_bytes(upload_widget):
    value = upload_widget.value
    if not value:
        return None, None
    if isinstance(value, dict):
        item = next(iter(value.values()))
    else:
        item = value[0]
    return item.get('name', 'voice-sample.wav'), bytes(item['content'])

def _get_voice_path():
    if voice_mode.value != 'reference':
        return None
    if not voice_consent.value:
        raise ValueError('Confirm that you own or have permission to use this recording and will not create a deceptive impersonation.')
    filename, content = _uploaded_bytes(voice_upload)
    if content:
        suffix = Path(filename).suffix.lower() or '.wav'
        path = VOICE_DIR / ('voice-' + uuid.uuid4().hex[:8] + suffix)
        path.write_bytes(content)
        state['voice_path'] = str(path)
        return str(path)
    if state['voice_path'] and Path(state['voice_path']).is_file():
        return state['voice_path']
    raise ValueError('Upload a short recording of your own voice first, or select the built-in Chatterbox voice.')

def _sync_storyboard():
    data = state['storyboard']
    if data is None:
        raise ValueError('Create a storyboard first.')
    for scene, widgets in zip(data['scenes'], state['scene_widgets']):
        scene['narration'] = widgets[0].value.strip()
        scene['on_screen_text'] = widgets[1].value.strip()
        scene['visual_prompt'] = widgets[2].value.strip()
    data['scenes'] = [s for s in data['scenes'] if s['narration']]
    data['full_narration'] = '\n'.join(s['narration'] for s in data['scenes'])
    return data

def _on_draft(_):
    with draft_output:
        clear_output()
        try:
            if provider.value:
                if not api_key.value.strip():
                    raise ValueError('Enter an API key for the selected provider, or select Template only.')
                save_provider_settings(provider.value, api_key.value, model_id.value)
            data = create_storyboard(topic.value, script.value, scene_notes.value, style.value, language.value,
                                     project_format.value, provider.value or None, model_id.value or None)
            state['storyboard'] = data
            state['scene_widgets'] = []
            cards = []
            for index, scene in enumerate(data['scenes']):
                narr = W.Textarea(description='Narration', value=scene.get('narration',''), layout=W.Layout(width='100%', height='90px'))
                onscreen = W.Textarea(description='Text', value=scene.get('on_screen_text',''), layout=W.Layout(width='100%', height='65px'))
                visual = W.Textarea(description='Visual', value=scene.get('visual_prompt',''), layout=W.Layout(width='100%', height='80px'))
                state['scene_widgets'].append((narr, onscreen, visual))
                cards.append(W.VBox([W.HTML(f'<b>Scene {index+1}</b>'), narr, onscreen, visual], layout=W.Layout(border='1px solid #888', padding='10px', margin='5px 0')))
            scene_editor.children = cards
            render_button.disabled = False
            display(Markdown(f'**Storyboard ready:** {len(data["scenes"])} scenes · mode: `{data["generation_mode"]}`'))
            if data.get('review_notes'):
                display(Markdown('**Review notes:** ' + ' · '.join(data['review_notes'])))
        except Exception as exc:
            display(Markdown('**Storyboard error:** ' + str(exc)))

def _on_preview(_):
    with voice_output:
        clear_output()
        try:
            if not torch.cuda.is_available():
                raise RuntimeError('No CUDA GPU was assigned. Select Runtime → Change runtime type → T4 GPU, if available, then reconnect.')
            if voice_mode.value == 'none':
                raise ValueError('Select a Chatterbox voice before previewing.')
            data = _sync_storyboard()
            ref = _get_voice_path()
            if not data['full_narration']:
                raise ValueError('The narration is empty.')
            preview_path = OUTPUTS / 'voice-preview.wav'
            info = synthesize(data['full_narration'][:250], preview_path, 'chatterbox', 'chatterbox-v3', 'hi', ref,
                              exaggeration.value, cfg_weight.value, temperature.value)
            display(Audio(filename=str(preview_path)))
            display(Markdown(f"Voice preview ready · {info['device']} · Chatterbox V3"))
        except Exception as exc:
            display(Markdown('**Voice preview error:** ' + str(exc)))

def _on_render(_):
    with render_output:
        clear_output()
        try:
            data = _sync_storyboard()
            ref = _get_voice_path() if voice_mode.value == 'reference' else None
            if visual_engine.value == 'animatediff' and (not torch.cuda.is_available() or GPU_VRAM < 12):
                raise RuntimeError('AnimateDiff requires a CUDA GPU with about 12 GB or more VRAM. This render will not silently fall back to motion cards.')
            if voice_mode.value != 'none' and not torch.cuda.is_available():
                raise RuntimeError('Chatterbox preview/render requires a Colab GPU in this notebook. Select No narration to use CPU-only rendering.')
            if voice_mode.value == 'reference' and not ref:
                raise ValueError('Upload your voice reference recording first.')
            display(Markdown('Rendering… Each AI footage scene is generated separately; this can take time on a free GPU.'))
            result = render_video(data, style.value, project_format.value,
                                  'chatterbox' if voice_mode.value != 'none' else 'none', 'chatterbox-v3', ref,
                                  exaggeration.value, cfg_weight.value, temperature.value,
                                  visual_engine.value, burn_captions.value, 'bold', duration.value, data.get('title'))
            state['result'] = result
            video_path = Path('/content/VIDEO-GENERATIONS-TOOLS') / 'workspace' / 'outputs' / result['job_id'] / 'video.mp4'
            persisted = EXPORT_DIR / (result['job_id'] + '-short-story.mp4')
            shutil.copy2(video_path, persisted)
            display(Video(str(video_path), embed=True, width=360))
            display(Markdown('**Finished files**'))
            for key in ('video_url','captions_url','timestamps_url','storyboard_url','manifest_url'):
                name = key.replace('_url','')
                relative = result[key].split('/outputs/',1)[-1]
                path = OUTPUTS / relative
                if path.exists():
                    drive_target = EXPORT_DIR / path.name if path.name != 'video.mp4' else persisted
                    if path.name != 'video.mp4':
                        shutil.copy2(path, drive_target)
                    display(FileLink(str(path)))
            display(Markdown(f'Video copied to Drive/session folder: `{persisted}`'))
        except Exception as exc:
            display(Markdown('**Render error:** ' + str(exc)))

draft_button.on_click(_on_draft)
preview_button.on_click(_on_preview)
render_button.on_click(_on_render)

display(W.VBox([
    W.HTML('<h2>01 · Script and storyboard</h2>'), topic, script, scene_notes,
    W.HBox([project_format, style, language]), W.HTML('<b>Optional LLM provider</b>'), provider, model_id, api_key, draft_button, draft_output, scene_editor,
    W.HTML('<hr><h2>02 · Voice preview</h2>'), voice_mode, voice_upload, voice_consent, exaggeration, cfg_weight, temperature, preview_button, voice_output,
    W.HTML('<hr><h2>03 · Video generation and export</h2>'), visual_engine, duration, burn_captions, render_button, render_output
]))


## Notes about model size and use
- Chatterbox Multilingual V3 is a 500M-parameter Hindi-capable voice model. It is loaded only when you preview/render audio.
- AnimateDiff-Lightning's 4-step adapter file is about 908 MB. It works with a Stable Diffusion 1.5 base checkpoint; total model downloads are several GB, but they remain in the Colab VM (or optional Hugging Face cache), not on your laptop or in GitHub.
- This video generator produces short clips per scene. It is not a replacement for a full-length video diffusion foundation model. Expect to inspect generated scenes and retry prompts when motion flickers or a scene is poor.
- The Stable Diffusion and AnimateDiff-Lightning checkpoints use CreativeML OpenRAIL-M terms; Chatterbox code/model uses MIT per upstream. The model licenses are not interchangeable with this repository's MIT app license.
- Do not upload anyone else's voice without permission. A reference sample should be clear, single-speaker, and ideally 5–10 seconds long.
- Colab may disconnect or decline to assign a GPU. That is a platform limit, not a reason to silently switch to another model. Download/export files before the session ends.
